Elimina las reacciones de intercambio (EX_) de los modelos AGORA2 antes de
pasarlos a Metage2Metabo.

MOTIVO
------
En formato COBRA una reaccion de intercambio se escribe sin productos:

    EX_dopa_e_ | dpmn_e <=>

Network expansion activa una reaccion cuando todos sus sustratos estan
disponibles. En la direccion inversa esta reaccion no tiene sustratos, asi que
la condicion se cumple siempre y el metabolito queda disponible gratis,
independientemente del seed.

Consecuencia verificada: con un seed de solo agua, proton y fosfato, m2m
reporta m-tiramina como producible, lo cual es estequiometricamente imposible.

Seed2LP elimina estas reacciones por defecto ("Import reaction: Removed").
m2m no lo hace, porque fue disenado para redes de Pathway Tools, que no las
traen. Este script hace ese preprocesamiento.

USO
    python preparar_sbml_para_m2m.py sbml/ sbml_sin_ex/

In [4]:
import sys
from pathlib import Path

import cobra


def limpia(entrada, salida):
    entrada, salida = Path(entrada), Path(salida)
    salida.mkdir(parents=True, exist_ok=True)

    for f in sorted(entrada.glob("*.xml")):
        try:
            modelo = cobra.io.read_sbml_model(str(f))
        except Exception as e:
            print(f"SALTADO  {f.name}: {type(e).__name__}")
            continue

        # Reacciones de frontera: introducen o retiran masa sin contraparte.
        # En network expansion la direccion inversa no tiene sustratos, asi que
        # se activa siempre y el metabolito queda disponible gratis.
        frontera = [r for r in modelo.reactions
                    if r.id.startswith(("EX_", "DM_", "SK_", "sink_"))]

        n_antes = len(modelo.reactions)
        # remove_orphans=False: conservamos los metabolitos extracelulares,
        # que son los que el seed introduce y los transportadores mueven.
        modelo.remove_reactions(frontera, remove_orphans=False)

        cobra.io.write_sbml_model(modelo, str(salida / f.name))
        print(f"OK  {f.name}: {n_antes} -> {len(modelo.reactions)} rxns "
              f"({len(frontera)} de frontera eliminadas)")


In [5]:
limpia("sbml/", "sbml_sin_ex/")

Set parameter Username
Set parameter LicenseID to value 2854521
Academic license - for non-commercial use only - expires 2027-08-19
OK  Akkermansia_muciniphila_ATCC_BAA_835_AGORA2_v2.01.xml: 1215 -> 1080 rxns (135 de frontera eliminadas)
OK  Bacteroides_fragilis_638R_AGORA2_v2.01.xml: 2286 -> 1776 rxns (510 de frontera eliminadas)
OK  Bacteroides_thetaiotaomicron_7330_AGORA2_v2.01.xml: 1504 -> 1297 rxns (207 de frontera eliminadas)
OK  Bifidobacterium_longum_DJO10A_AGORA2_v2.01.xml: 1984 -> 1464 rxns (520 de frontera eliminadas)
SALTADO  Clostridium_sporogenes_ATCC_15579_AGORA2_v2.01.xml: CobraSBMLError
OK  Eggerthella_lenta_A2_AGORA2_v2.01.xml: 1143 -> 1035 rxns (108 de frontera eliminadas)
OK  Eggerthella_lenta_DSM_11767_AGORA2_v2.01.xml: 1163 -> 1052 rxns (111 de frontera eliminadas)
OK  Eggerthella_lenta_DSM_2243_AGORA2_v2.01.xml: 1035 -> 900 rxns (135 de frontera eliminadas)
OK  Enterococcus_faecalis_ATCC_29212_AGORA2_v2.01.xml: 1560 -> 1375 rxns (185 de frontera eliminadas)
OK  E